# STM - Superior Tribunal Militar

Example notebook for the STM scraper, which searches the court's decisions in the [Consulta Judicial Unificada da Justiça Militar da União](https://jurisprudencia.stm.jus.br/).

| Feature | Available |
|---------|----------|
| cjsg    | Yes      |

The portal sits behind a WAF that refuses requests depending on the source IP address: it rejected every request from a datacenter IP address outside Brazil and accepted a residential one in Brazil (16 November 2026). When refused, `cjsg` raises `BotChallengeBlockedError` instead of returning an empty search. Run this notebook from a connection the portal accepts.

In [1]:
import juscraper as jus

stm = jus.scraper("stm")

## Basic search

Each page has 100 decisions, from the most recent judgment to the oldest. `paginas` is 1-based; `None` (default) downloads every page.

In [2]:
df = stm.cjsg("deserção", paginas=1)

print(df.shape)
df.head(3)

Baixando páginas STM:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas STM: 100%|██████████| 1/1 [00:00<00:00, 6052.39it/s]

(100, 12)


,processo,classe,relator,relator_acordao,revisor,assunto,data_autuacao,data_julgamento,data_publicacao,ementa,url_inteiro_teor,id_documento
0,7000339-36.2026.7.00.0000,RECURSO EM SENTIDO ESTRITO,GUIDO AMIN NAVES,None,None,"1) DIREITO PENAL MILITAR, CRIMES CONTRA O SERV...",2026-05-21,2026-10-15,2026-11-03,EMENTA: DIREITO PROCESSUAL PENAL MILITAR. RECU...,https://eproc2g.stm.jus.br/eproc_2g_prod/exter...,732eb247f354b6e9f416466a576d5f70
1,7000339-05.2024.7.03.0103,APELAÇÃO CRIMINAL,ARTUR VIDIGAL DE OLIVEIRA,None,CARLOS VUYK DE AQUINO,"1) DIREITO PENAL MILITAR, CRIMES CONTRA O SERV...",2025-11-11,2026-10-06,2026-11-05,EMENTA: \t\t\t\t\t\tDIREITO PENAL MILITAR. DIR...,https://eproc2g.stm.jus.br/eproc_2g_prod/exter...,edf2e7ffff99b9cc043f0181977d5ea3
2,7000530-81.2026.7.00.0000,EMBARGOS DE DECLARAÇÃO CRIMINAL,CARLOS VUYK DE AQUINO,None,None,"1) DIREITO PENAL MILITAR, CRIMES CONTRA O SERV...",2026-08-05,2026-09-24,2026-10-06,EMENTA: DIREITO PENAL MILITAR. DIREITO PROCESS...,https://eproc2g.stm.jus.br/eproc_2g_prod/exter...,74c026fd79a31535a2f3cb45677d595b


## Available columns

In [3]:
df.dtypes

processo            object
classe              object
relator             object
relator_acordao     object
revisor             object
assunto             object
data_autuacao       object
data_julgamento     object
data_publicacao     object
ementa              object
url_inteiro_teor    object
id_documento        object
dtype: object

In [4]:
print(df["ementa"].iloc[0][:300])

EMENTA: DIREITO PROCESSUAL PENAL MILITAR. RECURSO EM SENTIDO ESTRITO. DESERÇÃO. PEDIDO DE PRISÃO PREVENTIVA. INSTRUÇÃO PROVISÓRIA DE DESERÇÃO. ART. 255 DO CPPM. AUSÊNCIA DE PERICULUM LIBERTATIS CONCRETO. MEDIDA CAUTELAR EXCEPCIONAL. SUFICIÊNCIA DO TERMO DE DESERÇÃO COMO INSTRUMENTO DE CAPTURA. ART. 


## Search operators and fields

`pesquisa` takes the operators of the portal: `E` (the default between words), `OU`, `NÃO`, quotes for an exact phrase, and `*` and `?` as wildcards. `buscar_em` chooses where the term is searched: `"ementa"` (default), `"inteiro_teor"` (the full text of the acórdão, which includes the headnote) or `"numero"` (case number, complete or partial).

In [5]:
df_ou = stm.cjsg("peculato OU furto", paginas=1)
df_frase = stm.cjsg('"corrupção ativa"', buscar_em="inteiro_teor", paginas=1)

print(df_ou.shape, df_frase.shape)

Baixando páginas STM:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas STM: 100%|██████████| 1/1 [00:00<00:00, 16644.06it/s]

Baixando páginas STM:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas STM: 100%|██████████| 1/1 [00:00<00:00, 35848.75it/s]

(100, 12) (100, 12)


In [6]:
stm.cjsg("7000612-54.2022.7.00.0000", buscar_em="numero").drop(columns="ementa")

Baixando páginas STM:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas STM: 100%|██████████| 1/1 [00:00<00:00, 3934.62it/s]

,processo,classe,relator,relator_acordao,revisor,assunto,data_autuacao,data_julgamento,data_publicacao,url_inteiro_teor,id_documento
0,7000612-54.2022.7.00.0000,APELAÇÃO CRIMINAL,JOSÉ COÊLHO FERREIRA,ODILSON SAMPAIO BENZI,ODILSON SAMPAIO BENZI,"1) DIREITO PENAL MILITAR,CRIMES CONTRA INCOLUM...",2022-09-02,2023-08-17,2023-09-11,https://eproc2g.stm.jus.br/eproc_2g_prod/exter...,2d277ef92fdcbd8ffa22ebe083e824e7


## Filters

`classe`, `relator`, `relator_acordao`, `revisor` and `assunto` are the filters of the "Filtrar por" column of the portal. They match the exact value the portal shows, in upper case.

In [7]:
df_apelacao = stm.cjsg("deserção", classe="APELAÇÃO", paginas=1)

print(df_apelacao.shape)
df_apelacao["classe"].value_counts()

Baixando páginas STM:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas STM: 100%|██████████| 1/1 [00:00<00:00, 11915.64it/s]

(100, 12)


classe
APELAÇÃO    100
Name: count, dtype: int64

## Date range

`data_julgamento_inicio`/`data_julgamento_fim` and `data_publicacao_inicio`/`data_publicacao_fim` take `DD/MM/YYYY`, `YYYY-MM-DD` or a `date`.

The portal filters by date range only in its advanced search, so with these filters the search goes through it. The advanced search looks for `pesquisa` as an exact phrase in the field chosen by `buscar_em` and does not apply `E`, `OU`, `NÃO` or the wildcards. A term with more than one word outside quotes emits a `UserWarning`; put it in quotes to confirm the exact phrase.

In [8]:
df_datas = stm.cjsg(
    "deserção",
    data_julgamento_inicio="2025-03-01",
    data_julgamento_fim="2025-03-31",
)

print(df_datas.shape)
print(df_datas["data_julgamento"].min(), df_datas["data_julgamento"].max())
df_datas.head(3)

Baixando páginas STM:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas STM: 100%|██████████| 1/1 [00:00<00:00, 15141.89it/s]

(3, 12)
2025-03-13 2025-03-20


,processo,classe,relator,relator_acordao,revisor,assunto,data_autuacao,data_julgamento,data_publicacao,ementa,url_inteiro_teor,id_documento
0,7000636-14.2024.7.00.0000,EMBARGOS INFRINGENTES E DE NULIDADE,GUIDO AMIN NAVES,None,PÉRICLES AURÉLIO LIMA DE QUEIROZ,"1) DIREITO PENAL MILITAR,CRIMES CONTRA O SERVI...",2024-10-14,2025-03-20,2025-04-29,EMENTA: DIREITO PENAL MILITAR. DIREITO PROCESS...,https://eproc2g.stm.jus.br/eproc_2g_prod/exter...,a6078179a94b355081ab716d3fcffba7
1,7000128-29.2024.7.11.0011,APELAÇÃO CRIMINAL,ODILSON SAMPAIO BENZI,None,PÉRICLES AURÉLIO LIMA DE QUEIROZ,"1) DIREITO PENAL MILITAR,CRIMES CONTRA O SERVI...",2024-09-20,2025-03-20,2025-04-10,EMENTA: DIREITO PENAL MILITAR. DIREITO PROCESS...,https://eproc2g.stm.jus.br/eproc_2g_prod/exter...,6ac9201570c3b94e6a405e5a276626af
2,0000173-29.2016.7.08.0008,APELAÇÃO CRIMINAL,CLÁUDIO PORTUGAL DE VIVEIROS,None,PÉRICLES AURÉLIO LIMA DE QUEIROZ,"1) DIREITO PENAL MILITAR,CRIMES CONTRA O SERVI...",2024-10-10,2025-03-13,2025-03-28,EMENTA: \t\t\t\t\t\tDIREITO PROCESSUAL PENAL M...,https://eproc2g.stm.jus.br/eproc_2g_prod/exter...,78f6e480366950e80c4f7d5bfa4ecc93


An empty `pesquisa` returns everything in the range.

In [9]:
df_semana = stm.cjsg("", data_julgamento_inicio="10/03/2025", data_julgamento_fim="13/03/2025")

print(df_semana.shape)
df_semana["classe"].value_counts().head()

Baixando páginas STM:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas STM: 100%|██████████| 1/1 [00:00<00:00, 28532.68it/s]

(29, 12)


classe
APELAÇÃO CRIMINAL                  11
HABEAS CORPUS CRIMINAL              7
AGRAVO INTERNO CRIMINAL             6
RECURSO EM SENTIDO ESTRITO          3
EMBARGOS DE DECLARAÇÃO CRIMINAL     1
Name: count, dtype: int64

## Download and parse separately

`cjsg_download` returns the raw HTML of each page and `cjsg_parse` turns it into the DataFrame.

In [10]:
brutos = stm.cjsg_download("deserção", paginas=1)
df_parsed = stm.cjsg_parse(brutos)

print(len(brutos), df_parsed.shape)

Baixando páginas STM:   0%|          | 0/1 [00:00<?, ?it/s]

Baixando páginas STM: 100%|██████████| 1/1 [00:00<00:00, 19239.93it/s]

1 (100, 12)
